# Santander Customer Transaction Prediction
## Deep Learning Assignment — Student Notebook

**Competition:** Santander Customer Transaction Prediction  
**Platform:** Kaggle  
**Task type:** Binary classification on tabular data  
**Main framework:** PyTorch

Competition page:  
https://www.kaggle.com/competitions/santander-customer-transaction-prediction/overview

---

### Learning goals

By completing this notebook, you should be able to:

- understand a real-world binary classification problem;
- perform exploratory data analysis (EDA);
- detect and handle class imbalance;
- create a leakage-safe train/validation workflow;
- apply feature scaling correctly;
- use **SMOTE** only on the training partition;
- implement a custom PyTorch `Dataset`;
- build `DataLoader` objects;
- design an MLP for tabular data;
- implement training and validation functions;
- evaluate a classifier with ROC-AUC, PR-AUC, a confusion matrix, and classification metrics;
- analyze the decision threshold;
- generate probabilities for a Kaggle submission.

> **Important:** This notebook intentionally contains TODO cells instead of a full solution.

## Student Rules

1. Complete every **TODO** cell.
2. Do not use the Kaggle test set for model selection.
3. Split the labeled training data **before** fitting preprocessing steps.
4. Fit the scaler using the training partition only.
5. Apply **SMOTE only to the training partition**.
6. Keep the validation partition untouched.
7. Use predicted **probabilities** when calculating ROC-AUC.
8. Record short observations below important plots and results.
9. Set random seeds so that your experiment is reasonably reproducible.

## Suggested Workflow

1. Problem understanding  
2. Environment and reproducibility  
3. Load the data  
4. Data inspection and EDA  
5. Prepare features and target  
6. Train/validation split  
7. Scaling  
8. SMOTE  
9. PyTorch Dataset and DataLoader  
10. Neural network architecture  
11. Loss, optimizer, and training configuration  
12. Training function  
13. Validation function  
14. Training loop  
15. Learning curves  
16. Final validation evaluation  
17. ROC and Precision–Recall analysis  
18. Confusion matrix and classification report  
19. Threshold analysis  
20. Error analysis  
21. Test inference  
22. Kaggle submission  
23. Final reflection

# 1. Problem Understanding

**Task:** Read the competition description and explain the prediction problem in your own words.

Your explanation should identify:

- what one row represents;
- what the target variable means;
- whether the task is regression, multi-class classification, or binary classification;
- what type of prediction the final model should produce.

**Hint:** A strong answer is short and precise. Do not copy the Kaggle description word-for-word.

In [ ]:
# One row represents one customer and the customer's anonymized transaction features.
# The target is 1 if the customer made a particular transaction and 0 otherwise.
# This is a binary classification problem.
# The model should output a probability between 0 and 1 for the positive class.


### 1.1 Evaluation Metric

**Task:** Find the competition evaluation metric and explain what it measures.

**Hint:** For an imbalanced binary classification problem, accuracy alone can be misleading.  
Also think about why a ranking-based metric can be useful.

In [ ]:
# Main evaluation metric: ROC-AUC (Area Under the Receiver Operating Characteristic Curve).
# A higher ROC-AUC means the model is better at ranking positive examples
# above negative examples across different classification thresholds.


# 2. Environment Setup

**Task:** Import the libraries required for data processing, visualization, preprocessing, SMOTE, PyTorch, and evaluation.

**Hint:** You will probably need packages from:
`numpy`, `pandas`, `matplotlib`, `sklearn`, `imblearn`, and `torch`.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, roc_curve, precision_recall_curve, confusion_matrix,
    classification_report
)
from imblearn.over_sampling import SMOTE

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


### 2.1 Reproducibility

**Task:** Set random seeds for Python, NumPy, and PyTorch.

**Hint:** If CUDA is available, remember that PyTorch also has CUDA-specific seeds.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


### 2.2 Select the Compute Device

**Task:** Select GPU when CUDA is available; otherwise use CPU.

**Hint:** `torch.cuda.is_available()` can be used to detect CUDA.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)


# 3. Load the Dataset

**Task:** Define file paths and load:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

**Hint:** On Kaggle, competition data is usually mounted under `/kaggle/input/...`.  
If you run locally, change the paths accordingly.

In [ ]:
# Use Kaggle's folder when available; otherwise use the current folder.
DATA_DIR = "/kaggle/input/santander-customer-transaction-prediction"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUBMISSION_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

print(TRAIN_PATH)
print(TEST_PATH)
print(SAMPLE_SUBMISSION_PATH)


### 3.1 Read CSV Files

**Task:** Load the three CSV files into pandas DataFrames.

**Hint:** Use clear variable names such as `train_df`, `test_df`, and `submission_df`.

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)


### 3.2 Verify That the Data Loaded Correctly

**Task:** Display the first few rows and print the shape of each DataFrame.

**Hint:** Do not print the complete dataset.

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Submission shape:", submission_df.shape)

display(train_df.head())
display(test_df.head())
display(submission_df.head())


# 4. Exploratory Data Analysis (EDA)

## 4.1 Columns and Data Types

**Task:** Inspect column names, data types, and basic dataset information.

**Hint:** Identify:
- the ID column;
- the target column;
- feature columns;
- non-numeric columns, if any.

In [ ]:
print(train_df.info())
print("\nTrain columns:", list(train_df.columns[:10]), "...")
print("ID column: ID_code")
print("Target column: target")
print("Non-numeric columns:", train_df.select_dtypes(exclude=np.number).columns.tolist())


## 4.2 Missing Values

**Task:** Check the number and percentage of missing values.

**Hint:** Report the columns with the most missing values first.  
If there are no missing values, explicitly state that observation.

In [ ]:
missing = train_df.isna().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
missing_table = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_table.sort_values("missing_count", ascending=False).head(10))


## 4.3 Duplicate Rows and Duplicate IDs

**Task:** Check for duplicated rows and duplicated identifiers.

**Hint:** A unique ID column should normally not contain duplicates.

In [ ]:
print("Duplicated rows:", train_df.duplicated().sum())
print("Duplicated IDs:", train_df["ID_code"].duplicated().sum())


## 4.4 Target Distribution

**Task:** Examine the class distribution of the target.

Create:

1. class counts;
2. class percentages;
3. a simple bar chart.

**Hint:** Calculate the minority-to-majority ratio.  
Ask yourself whether accuracy would be a trustworthy metric here.

In [ ]:
class_counts = train_df["target"].value_counts().sort_index()
class_pct = train_df["target"].value_counts(normalize=True).sort_index().mul(100).round(2)
print("Class counts:\n", class_counts)
print("\nClass percentages:\n", class_pct)
print("\nMinority/Majority ratio:", class_counts.min() / class_counts.max())

class_counts.plot(kind="bar", title="Target Distribution")
plt.xlabel("Target")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()


### Your Observation

Write 2–4 sentences describing the class balance.

**Hint:** Mention which class is the minority class and why this matters during training.

In [ ]:
# The target is strongly imbalanced: class 1 is the minority class and class 0 is the majority class.
# Because of this imbalance, accuracy alone can be misleading.
# ROC-AUC, PR-oriented metrics, and minority-class recall are more informative.


## 4.5 Descriptive Statistics

**Task:** Generate descriptive statistics for the numerical features.

**Hint:** Do not manually inspect every feature. Look for unusual ranges, scales, means, or standard deviations.

In [ ]:
feature_cols = [c for c in train_df.columns if c not in ["ID_code", "target"]]
print(train_df[feature_cols].describe().T.head(15))


## 4.6 Feature Distributions

**Task:** Select a small subset of features and visualize their distributions.

Compare distributions for the two target classes when useful.

**Hint:** Plot only a manageable number of features (for example 4–8).  
Possible plots: histogram, KDE-like density plot, or box plot.

In [ ]:
sample_features = feature_cols[:4]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes = axes.ravel()
for ax, col in zip(axes, sample_features):
    ax.hist(train_df.loc[train_df["target"] == 0, col], bins=30, alpha=0.5, label="target=0", density=True)
    ax.hist(train_df.loc[train_df["target"] == 1, col], bins=30, alpha=0.5, label="target=1", density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()


## 4.7 Correlation Exploration

**Task:** Investigate correlations among features and between features and the target.

**Hint:** With many features, a full correlation heatmap may be unreadable.  
Consider selecting the strongest correlations or a small subset of features.

In [ ]:
corr = train_df[feature_cols + ["target"]].corr(numeric_only=True)["target"].drop("target").abs().sort_values(ascending=False)
print("Top 10 absolute feature-target correlations:")
print(corr.head(10))

train_df[corr.head(10).index.tolist() + ["target"]].corr(numeric_only=True)["target"].drop("target").sort_values().plot(kind="barh", figsize=(8, 5), title="Top Feature-Target Correlations")
plt.xlabel("Correlation")
plt.show()


# 5. Prepare Features and Target

**Task:** Separate the feature matrix `X` and target vector `y`.

The identifier should not be used as a predictive feature unless you can justify it.

**Hint:** Keep the test IDs separately because you will need them for submission.

In [ ]:
target_col = "target"
id_col = "ID_code"

X = train_df.drop(columns=[id_col, target_col]).copy()
y = train_df[target_col].astype(int).copy()
test_ids = test_df[id_col].copy()
X_test = test_df.drop(columns=[id_col]).copy()


### 5.1 Sanity Checks

**Task:** Confirm that feature columns in training and test data match.

**Hint:** Compare column names and column order, not only the number of columns.

In [ ]:
assert list(X.columns) == list(X_test.columns), "Train and test feature columns do not match."
print("Number of matching features:", len(X.columns))
print("Feature order matches: True")


# 6. Train / Validation Split

**Task:** Split the labeled data into training and validation partitions.

Requirements:

- use a fixed random seed;
- preserve the class ratio;
- keep the validation set untouched for evaluation.

**Hint:** Use a **stratified** split.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print(X_train.shape, X_val.shape)


### 6.1 Verify the Split

**Task:** Compare the target distribution in the training and validation partitions.

**Hint:** Their positive-class percentages should be similar when stratification is used.

In [ ]:
print("Training class percentages:")
print(y_train.value_counts(normalize=True).sort_index().mul(100).round(2))
print("\nValidation class percentages:")
print(y_val.value_counts(normalize=True).sort_index().mul(100).round(2))


# 7. Feature Scaling

**Task:** Standardize the input features.

Rules:

- fit the scaler on `X_train` only;
- transform `X_train`;
- transform `X_val` using the same fitted scaler;
- later, transform Kaggle test features using that same scaler.

**Hint:** Fitting the scaler on the full dataset causes data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


### 7.1 Scaling Check

**Task:** Verify that scaling behaved as expected.

**Hint:** Inspect the approximate mean and standard deviation of a few scaled training features.

In [ ]:
print("First 5 feature means after scaling:")
print(np.mean(X_train_scaled[:, :5], axis=0))
print("First 5 feature stds after scaling:")
print(np.std(X_train_scaled[:, :5], axis=0))


# 8. Handle Class Imbalance with SMOTE

**Task:** Apply SMOTE to the **training partition only**.

Do **not** apply SMOTE to validation data.

**Hint:** Use `imblearn.over_sampling.SMOTE`.  
Inspect the class distribution before and after resampling.

> Think about the `sampling_strategy` you choose. Full 1:1 balancing is not always automatically optimal.

In [ ]:
smote = SMOTE(sampling_strategy=0.5, random_state=SEED)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print("Before SMOTE:", y_train.value_counts().sort_index().to_dict())
print("After SMOTE:", pd.Series(y_train_resampled).value_counts().sort_index().to_dict())


### 8.1 Compare Class Counts Before and After SMOTE

**Task:** Show how SMOTE changed the training class distribution.

**Hint:** A small table or bar chart is enough.

In [ ]:
before = y_train.value_counts().sort_index()
after = pd.Series(y_train_resampled).value_counts().sort_index()
compare = pd.DataFrame({"Before_SMOTE": before, "After_SMOTE": after})
print(compare)
compare.plot(kind="bar", title="Class Counts Before and After SMOTE")
plt.xlabel("Target")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()


### 8.2 Short Reflection on SMOTE

Answer briefly:

- Why must SMOTE be applied after the split?
- Why should validation data remain untouched?
- What possible downside can synthetic oversampling introduce?

**Hint:** Think about leakage, realism of synthetic points, and overfitting.

In [ ]:
# SMOTE must be applied after the train/validation split so synthetic samples cannot use validation information.
# The validation set stays untouched so it remains a realistic estimate of generalization.
# A downside is that synthetic points may not perfectly represent real customers and can sometimes increase overfitting.


# 9. Convert Data to PyTorch-Friendly Format

**Task:** Convert the resampled training data and untouched validation data to appropriate numeric types.

**Hint:** Neural-network inputs are usually `float32`.  
For `BCEWithLogitsLoss`, binary targets are also typically floating-point values.

In [ ]:
X_train_resampled = np.asarray(X_train_resampled, dtype=np.float32)
y_train_resampled = np.asarray(y_train_resampled, dtype=np.float32)
X_val_scaled = np.asarray(X_val_scaled, dtype=np.float32)
y_val = np.asarray(y_val, dtype=np.float32)


# 10. Create a Custom PyTorch Dataset

**Task:** Implement a custom `Dataset` class for tabular binary classification.

Your class should implement:

- `__init__`
- `__len__`
- `__getitem__`

**Hint:** Each sample should return a feature tensor and its target tensor.

In [ ]:
class SantanderDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


### 10.1 Instantiate Dataset Objects

**Task:** Create training and validation dataset objects.

**Hint:** Training should use the SMOTE-resampled data; validation must use the original validation split.

In [ ]:
train_dataset = SantanderDataset(X_train_resampled, y_train_resampled)
val_dataset = SantanderDataset(X_val_scaled, y_val)


# 11. Create DataLoaders

**Task:** Build `DataLoader` objects for training and validation.

Think about:

- batch size;
- shuffling;
- number of workers;
- whether the last incomplete batch should be dropped.

**Hint:** Training data is normally shuffled. Validation data normally is not.

In [ ]:
BATCH_SIZE = 256

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


### 11.1 Inspect One Batch

**Task:** Read one batch and inspect feature and target shapes.

**Hint:** Confirm that the input shape is compatible with the first layer of your network.

In [ ]:
xb, yb = next(iter(train_loader))
print("Feature batch shape:", xb.shape)
print("Target batch shape:", yb.shape)


# 12. Build the Neural Network

**Task:** Implement an MLP for tabular binary classification.

Consider using:

- several fully connected layers;
- nonlinear activations;
- dropout;
- optional batch normalization;
- one output logit.

**Hint:** If you use `BCEWithLogitsLoss`, do **not** place a sigmoid layer inside the final model output.

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


### 12.1 Instantiate and Inspect the Model

**Task:** Create the model, move it to the selected device, and inspect the number of trainable parameters.

**Hint:** The input dimension should be derived from the feature matrix rather than hard-coded when possible.

In [ ]:
model = TabularMLP(X_train_resampled.shape[1]).to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


# 13. Loss Function, Optimizer, and Training Configuration

**Task:** Define:

- loss function;
- optimizer;
- learning rate;
- number of epochs;
- optional weight decay;
- optional learning-rate scheduler.

**Hint:** `BCEWithLogitsLoss` is a natural choice for a single-logit binary classifier.  
`Adam` or `AdamW` are reasonable starting optimizers.

In [ ]:
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)


# 14. Implement One Training Epoch

**Task:** Write a function that trains the model for one epoch.

Your function should:

1. switch the model to training mode;
2. iterate over batches;
3. move data to the device;
4. clear previous gradients;
5. perform the forward pass;
6. compute loss;
7. backpropagate;
8. update model parameters;
9. accumulate average training loss.

**Hint:** Pay attention to the target shape so that it matches the model output.

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    total = 0

    for X_batch, y_batch in data_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = X_batch.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / total


# 15. Implement Validation

**Task:** Write a validation function.

It should:

- switch the model to evaluation mode;
- disable gradient calculation;
- compute validation loss;
- convert logits to probabilities;
- collect all true labels and probabilities;
- calculate ROC-AUC.

**Hint:** Apply sigmoid to logits during evaluation to obtain probabilities.

In [ ]:
def validate_one_epoch(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0
    y_true = []
    y_prob = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            probs = torch.sigmoid(logits)
            batch_size = X_batch.size(0)
            running_loss += loss.item() * batch_size
            total += batch_size
            y_true.extend(y_batch.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())

    avg_loss = running_loss / total
    auc = roc_auc_score(y_true, y_prob)
    return avg_loss, auc, np.asarray(y_true), np.asarray(y_prob)


# 16. Training Loop

**Task:** Train the model for multiple epochs.

Store at least:

- training loss;
- validation loss;
- validation ROC-AUC.

Print a compact progress message for every epoch.

**Hint:** Keep metric history in lists or a dictionary so that you can plot it later.

In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": [],
}

best_auc = -np.inf
best_state = None
patience = 3
bad_epochs = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_auc, _, _ = validate_one_epoch(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)

    if val_auc > best_auc:
        best_auc = val_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    scheduler.step(val_auc)
    print(f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_auc={val_auc:.4f}")

    if bad_epochs >= patience:
        print("Early stopping.")
        break

model.load_state_dict(best_state)
print(f"Best validation ROC-AUC: {best_auc:.4f}")


### 16.1 Optional: Early Stopping

**Bonus Task:** Stop training when validation ROC-AUC no longer improves.

**Hint:** Save the best model parameters whenever validation AUC improves.  
Use a patience counter.

In [ ]:
# Bonus checkpoint/early-stopping work is already included in the training loop above.
print("Best model checkpoint restored; early stopping was used.")


# 17. Plot Learning Curves

**Task:** Plot:

1. training loss vs. epoch;
2. validation loss vs. epoch;
3. validation ROC-AUC vs. epoch.

**Hint:** Use separate readable plots and label the axes clearly.

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs, history["val_auc"], label="Validation ROC-AUC")
plt.xlabel("Epoch")
plt.ylabel("ROC-AUC")
plt.title("Validation ROC-AUC")
plt.legend()
plt.show()


### Your Training Observation

Write a short interpretation.

Consider:

- Did the model improve consistently?
- Is there evidence of overfitting?
- At which epoch was validation performance best?

**Hint:** Base your comments on the plots rather than assumptions.

In [ ]:
# The best epoch is the one with the highest validation ROC-AUC.
# A growing training/validation loss gap together with falling validation AUC would indicate overfitting.
# Early stopping and the best checkpoint help limit this problem.


# 18. Final Validation Predictions

**Task:** Generate final probabilities for the validation set using the best available model.

**Hint:** Keep both:
- predicted probability;
- predicted class at a default threshold of 0.50.

In [ ]:
model.eval()
with torch.no_grad():
    val_logits = model(torch.tensor(X_val_scaled, dtype=torch.float32).to(device))
    y_val_prob = torch.sigmoid(val_logits).cpu().numpy()

y_val_true = y_val.astype(int)
y_val_pred_05 = (y_val_prob >= 0.50).astype(int)


# 19. Main Validation Metrics

**Task:** Calculate several metrics.

Required:

- ROC-AUC;
- Average Precision / PR-AUC-style summary;
- precision;
- recall;
- F1-score.

**Hint:** ROC-AUC uses probabilities.  
Precision, recall, and F1 require a decision threshold.

In [ ]:
roc_auc = roc_auc_score(y_val_true, y_val_prob)
pr_auc = average_precision_score(y_val_true, y_val_prob)
precision = precision_score(y_val_true, y_val_pred_05, zero_division=0)
recall = recall_score(y_val_true, y_val_pred_05, zero_division=0)
f1 = f1_score(y_val_true, y_val_pred_05, zero_division=0)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Average Precision (PR-AUC style): {pr_auc:.4f}")
print(f"Precision @ 0.50: {precision:.4f}")
print(f"Recall @ 0.50: {recall:.4f}")
print(f"F1 @ 0.50: {f1:.4f}")


# 20. ROC Curve

**Task:** Plot the ROC curve and report ROC-AUC.

**Hint:** Plot the random-classifier diagonal as a reference line.

In [ ]:
fpr, tpr, _ = roc_curve(y_val_true, y_val_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], "--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


# 21. Precision–Recall Curve

**Task:** Plot the precision–recall curve.

**Hint:** PR analysis is especially informative when the positive class is relatively rare.

In [ ]:
precisions, recalls, _ = precision_recall_curve(y_val_true, y_val_prob)
plt.figure(figsize=(7, 5))
plt.plot(recalls, precisions, label=f"AP = {pr_auc:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()


# 22. Confusion Matrix

**Task:** Create a confusion matrix using threshold = 0.50.

Label the four outcomes:

- True Negative
- False Positive
- False Negative
- True Positive

**Hint:** Do not judge the model from the confusion matrix alone; the chosen threshold directly affects it.

In [ ]:
cm = confusion_matrix(y_val_true, y_val_pred_05)
print("Confusion matrix:\n", cm)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix @ 0.50")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks([0, 1], ["0", "1"])
plt.yticks([0, 1], ["0", "1"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar()
plt.show()


# 23. Classification Report

**Task:** Print a classification report at threshold = 0.50.

**Hint:** Compare precision, recall, and F1 for the minority class with the majority class.

In [ ]:
print(classification_report(y_val_true, y_val_pred_05, digits=4, zero_division=0))


# 24. Decision Threshold Analysis

**Task:** Evaluate several thresholds instead of assuming 0.50 is always best.

For each threshold, calculate at least:

- precision;
- recall;
- F1-score.

**Hint:** Try a grid of threshold values and visualize how the metrics change.

> Note: A decision threshold affects class labels and the confusion matrix, but ROC-AUC is calculated from ranking/probabilities and does not require a fixed classification threshold.

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
threshold_results = []

for threshold in thresholds:
    pred = (y_val_prob >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val_true, pred, zero_division=0),
        "recall": recall_score(y_val_true, pred, zero_division=0),
        "f1": f1_score(y_val_true, pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)
threshold_df.set_index("threshold")[["precision", "recall", "f1"]].plot(figsize=(8, 5))
plt.ylabel("Score")
plt.title("Metrics at Different Thresholds")
plt.show()


### 24.1 Choose a Threshold

**Task:** Select a threshold based on a clearly stated objective.

**Hint:** Examples:
- maximize F1;
- prioritize recall;
- prioritize precision.

There is no universally best threshold without a business objective.

In [ ]:
best_threshold = float(threshold_df.loc[threshold_df["f1"].idxmax(), "threshold"])
print(f"Chosen threshold: {best_threshold:.2f}")
print("Reason: it gives the highest validation F1-score, balancing precision and recall.")


# 25. Error Analysis

**Task:** Inspect some false positives and false negatives.

**Hint:** Create a small DataFrame containing:
- true label;
- predicted probability;
- predicted class;
- original feature values or selected informative features.

Look for patterns rather than listing hundreds of rows.

In [ ]:
val_errors = X_val.reset_index().copy()
val_errors["true"] = y_val_true
val_errors["probability"] = y_val_prob
val_errors["predicted"] = (y_val_prob >= best_threshold).astype(int)

false_positives = val_errors[(val_errors["true"] == 0) & (val_errors["predicted"] == 1)].copy()
false_negatives = val_errors[(val_errors["true"] == 1) & (val_errors["predicted"] == 0)].copy()

print("False positives:")
display(false_positives[["index", "true", "probability", "predicted"] + feature_cols[:5]].head(10))
print("False negatives:")
display(false_negatives[["index", "true", "probability", "predicted"] + feature_cols[:5]].head(10))


### Error Analysis Observation

Write 3–5 sentences describing what you learned from the mistakes.

**Hint:** Mention model confidence and whether some errors occur near the chosen threshold.

In [ ]:
# Many mistakes near the chosen threshold are expected because those cases are less separable.
# False positives are negative examples that the model considered sufficiently likely to be positive.
# False negatives are positive examples whose predicted probability stayed below the threshold.
# Inspecting probabilities helps distinguish confident errors from borderline errors.


# 26. Compare Imbalance Strategies

**Recommended Experiment**

Train or evaluate at least two approaches:

- **Model A:** training without SMOTE;
- **Model B:** training with SMOTE.

Optional third approach:

- **Model C:** no SMOTE, but use class weighting / `pos_weight` in the loss.

**Hint:** Use the same validation split for a fair comparison.  
Compare ROC-AUC, PR-oriented metrics, and minority-class recall—not only accuracy.

In [ ]:
# Fair comparison: train a second model with the same split, scaling, architecture and optimizer,
# but WITHOUT SMOTE. A few epochs are enough for a simple baseline comparison.
COMPARE_EPOCHS = min(5, NUM_EPOCHS)

plain_train_dataset = SantanderDataset(np.asarray(X_train_scaled, dtype=np.float32), np.asarray(y_train, dtype=np.float32))
plain_train_loader = DataLoader(plain_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

plain_model = TabularMLP(X_train_scaled.shape[1]).to(device)
plain_criterion = nn.BCEWithLogitsLoss()
plain_optimizer = torch.optim.AdamW(plain_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

for epoch in range(COMPARE_EPOCHS):
    loss_a = train_one_epoch(plain_model, plain_train_loader, plain_criterion, plain_optimizer, device)
    val_loss_a, auc_a, true_a, prob_a = validate_one_epoch(plain_model, val_loader, plain_criterion, device)
    print(f"No-SMOTE epoch {epoch+1}/{COMPARE_EPOCHS}: train_loss={loss_a:.4f}, val_auc={auc_a:.4f}")

plain_pred = (prob_a >= 0.50).astype(int)
comparison_results = [
    {
        "Experiment": "No SMOTE",
        "SMOTE": "No",
        "Class Weight": "No",
        "Val ROC-AUC": roc_auc_score(true_a, prob_a),
        "Precision": precision_score(true_a, plain_pred, zero_division=0),
        "Recall": recall_score(true_a, plain_pred, zero_division=0),
        "F1": f1_score(true_a, plain_pred, zero_division=0),
    },
    {
        "Experiment": "SMOTE",
        "SMOTE": "Yes",
        "Class Weight": "No",
        "Val ROC-AUC": roc_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    },
]


### 26.1 Comparison Table

**Task:** Create a compact comparison table.

Suggested columns:

| Experiment | SMOTE | Class Weight | Val ROC-AUC | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|

**Hint:** Keep all other important settings as similar as possible.

In [ ]:
comparison_df = pd.DataFrame(comparison_results)
display(comparison_df.round(4))


# 27. Prepare Kaggle Test Features

**Task:** Apply the already-fitted preprocessing pipeline to the Kaggle test data.

**Hint:** Never fit a new scaler on test data.

In [ ]:
X_test_scaled = scaler.transform(X_test).astype(np.float32)


# 28. Test-Set Inference

**Task:** Generate predicted probabilities for every row in the Kaggle test set.

**Hint:** Use evaluation mode and disable gradients.  
Your output should contain one probability per test row.

In [ ]:
test_dataset = torch.tensor(X_test_scaled, dtype=torch.float32)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_probabilities = []
model.eval()
with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        probs = torch.sigmoid(logits)
        test_probabilities.extend(probs.cpu().numpy())

test_probabilities = np.asarray(test_probabilities).reshape(-1)
print("Test probabilities shape:", test_probabilities.shape)


# 29. Create the Kaggle Submission File

**Task:** Create a submission DataFrame in the required Kaggle format.

**Hint:** Start from `sample_submission.csv` when possible so that the column names and order remain correct.  
For ROC-AUC competitions, submit probabilities rather than thresholded 0/1 class labels.

In [ ]:
submission = submission_df.copy()
prediction_col = target_col if target_col in submission.columns else submission.columns[-1]
submission[prediction_col] = test_probabilities
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)


### 29.1 Submission Sanity Checks

**Task:** Before saving or uploading, verify:

- row count matches the test set;
- ID order is correct;
- probability values are finite;
- probabilities lie between 0 and 1;
- no missing values exist;
- required columns are present.

**Hint:** Submission-format mistakes can invalidate an otherwise good model.

In [ ]:
assert len(submission) == len(test_df)
assert len(test_ids) == len(test_probabilities)
assert np.isfinite(test_probabilities).all()
assert ((test_probabilities >= 0) & (test_probabilities <= 1)).all()
assert submission.isna().sum().sum() == 0
assert set(submission_df.columns).issubset(submission.columns)
assert submission[id_col].equals(submission_df[id_col]) if id_col in submission.columns and id_col in submission_df.columns else True
print("All submission sanity checks passed.")


# 30. Final Reflection

Answer briefly in Markdown.

1. What was the validation ROC-AUC of your best model?
2. Did SMOTE improve performance? Which metrics changed?
3. Did the model overfit?
4. Which threshold did you choose for class-label analysis, and why?
5. Which evaluation metric was most informative for this problem?
6. What is one limitation of your current neural-network approach?
7. What would you try next if you had more time?

In [ ]:
# 1. The best validation ROC-AUC is printed in the training section and stored in `best_auc`.
# 2. The SMOTE comparison is shown in `comparison_df`; use the ROC-AUC, PR-oriented metrics, recall and F1 to judge the effect.
# 3. Overfitting is checked with the learning curves and controlled with early stopping.
# 4. The threshold for class-label analysis is the threshold with the highest validation F1.
# 5. ROC-AUC is the main competition metric, while Average Precision is especially useful for the imbalanced positive class.
# 6. A limitation is that the MLP may still miss useful structure that tree-based models can capture in tabular data.
# 7. Next steps could include cross-validation, stronger hyperparameter tuning, and comparison with a non-neural baseline.


# Bonus Challenges

Choose one or more:

### A. Learning-Rate Scheduler
Add a scheduler and compare the learning curves.

### B. Batch Normalization
Compare a model with and without batch normalization.

### C. Dropout Study
Try multiple dropout rates and report validation ROC-AUC.

### D. Hidden-Layer Architecture
Compare a shallow and a deeper MLP.

### E. Cross-Validation
Use stratified K-fold validation and report mean and standard deviation of ROC-AUC.

### F. Probability Calibration
Investigate whether predicted probabilities are well calibrated.

### G. Feature Ablation
Train using subsets of features and compare performance.

### H. Model Explainability
Use an appropriate explainability method to investigate which features influence predictions.

### I. Baseline Comparison
Compare the MLP against a simple non-neural baseline such as logistic regression.

**Hint:** Change one major factor at a time so that your comparison remains interpretable.

In [ ]:
# Bonus challenge: a learning-rate scheduler was added in the optimizer section using ReduceLROnPlateau.
# It reduces the learning rate when validation ROC-AUC stops improving.


# Completion Checklist

Before submitting your notebook, confirm that you have completed all of the following:

- [ ] Problem described correctly
- [ ] Data loaded and inspected
- [ ] Missing values checked
- [ ] Duplicates checked
- [ ] Target imbalance analyzed
- [ ] Useful EDA plots created
- [ ] ID removed from model features
- [ ] Stratified train/validation split created
- [ ] Scaler fitted only on training data
- [ ] SMOTE applied only to training data
- [ ] Custom PyTorch Dataset implemented
- [ ] Train and validation DataLoaders created
- [ ] MLP model implemented
- [ ] Training function implemented
- [ ] Validation function implemented
- [ ] ROC-AUC tracked during training
- [ ] Learning curves plotted
- [ ] ROC curve plotted
- [ ] Precision–Recall curve plotted
- [ ] Confusion matrix plotted
- [ ] Classification report generated
- [ ] Threshold analysis completed
- [ ] Error analysis completed
- [ ] SMOTE comparison completed
- [ ] Kaggle test inference completed
- [ ] Submission file validated
- [ ] Final reflection written